### vision 모델 내부 동작 원리
- 이미지 인코딩:이미지 임베딩 벡터
- 프로젝션: 이미지 벡터를 LLM 토큰 공간으로 변환
- 
- LLM
- 

In [1]:
from langchain_core.output_parsers import StrOutputParser

from langchain_core.prompts import ChatPromptTemplate

from langchain_core.documents import Document

from langchain_core.messages import HumanMessage

from langchain_ibm import WatsonxEmbeddings
from langchain_ibm import ChatWatsonx

from langchain_ollama import ChatOllama

from langchain_chroma import Chroma

from pathlib import Path

from dotenv import load_dotenv

import base64

import os

c:\souce\ollama\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [27]:
#.env 내용 가죠오기
load_dotenv()

apikey = os.getenv("WATSONX_API_KEY")
project_id = os.getenv("WATSONX_PROJECT_ID")
watsonx_ai_url = os.getenv("WATSONX_URL")
hf_token = os.getenv("HF_TOKEN")
cohere_api_key = os.getenv("COHERE_API_KEY")
serper_api_key = os.getenv("SERPER_API_KEY")

# 유료 LLM 선언

watson_llm = ChatWatsonx(
    model_id="ibm/granite-4-h-small",
    url = f"{watsonx_ai_url}",
    api_key = f"{apikey}",
    project_id=f"{project_id}",
    params = {
    "max_tokens": 2000,
    "temperature": 0
    }
)

# 로컬 LLM 선언
qwen_llm = ChatOllama(model="qwen3.5:4b",temperature= 0)

exaone_llm = ChatOllama(model="exaone3.5:2.4b",temperature= 0)

watsonx_embedding = WatsonxEmbeddings(
    model_id="ibm/granite-embedding-278m-multilingual",
    url = f"{watsonx_ai_url}",
    api_key = f"{apikey}",
    project_id=f"{project_id}"
)

In [3]:
# image => base64 인코딩
def encode_image(image_path:str)->str:
    with open(image_path,"rb") as f:
        return base64.b64encode(f.read()).decode('utf-8')

In [6]:
# ivision model
vision_llm = ChatOllama(model="minimax-m3:cloud",temperature=0)

parsor = StrOutputParser()
# 이미지+텍스트 : 이미지를 텍스트형식으로 질문
def ask_about_image(image_path, question):
    image_b64 = encode_image(image_path)
    prompt = ChatPromptTemplate.from_messages(
        [
            ("human",[
                {"type":"image_url","image_url":{'url':f'data:image/jpeg;base64,{image_b64}'}},
                {'type':"text", "text":question}
            ])
        ]
    )

    chain = prompt | vision_llm | parsor
    response = chain.invoke({"image_b64":image_b64, "question":question})

    return response

In [15]:
ask_about_image("./image/animals1.jpg","이 이미지에서 무엇이 보이나요?")

'이 이미지에는 **고양이**가 보입니다. 구체적으로 살펴보면:\n\n- **얼굴이 카메라를 정면으로 응시하는 클로즈업 샷**입니다\n- **터비 패턴(줄무늬)**의 갈색과 검은색 털을 가지고 있습니다\n- **선명한 청록색/파란색 눈**이 매우 인상적입니다\n- **긴 흰색 수염**이 양옆으로 펼쳐져 있습니다\n- 입이 살짝 벌어져 있어 **야옹하거나 울려고 하는 모습**처럼 보입니다\n- **분홍색 코**가 가운데 자리 잡고 있습니다\n- **배경은 따뜻한 오렌지/갈색 톤의 나무 바닥**으로 보입니다\n\n위에서 아래로 내려다본 각도에서 촬영된, 고양이의 표정과 시선이 잘 살아있는 귀여운 사진입니다. 🐱'

#### 2. URL 이미지

In [19]:
import requests
from io import BytesIO
from PIL import Image

In [ ]:
# 이미지 다운로드
url = "https://search.pstatic.net/common/?src=http%3A%2F%2Fblogfiles.naver.net%2FMjAyNjAyMDZfMjYw%2FMDAxNzcwMzYxMzIzMDM0.nI0o7cWscvcH52S9d6c7_71gcnGP4RjfsxxLIuQvmUQg.9FqWVXR4K5G67uc7FJFD1zdCDL0F5pXFXs1T9aTaJU8g.JPEG%2F5.jpg&type=a340"
img = requests.get(url).content
# 인코딩
img_b64= base64.b64encode(img).decode()
# 모델
message = HumanMessage(
    content= [
        {"type":"image_url","image_url":{'url':f'data:image/jpeg;base64,{img_b64}'}},
        {'type':"text", "text":" 이 동물들의 종류와 특징을 말해주세요."}
    ]
)

vision_llm.invoke([message]).content

'# 토끼 (Rabbit)\n\n이미지에 보이는 동물들은 **토끼**입니다. 봄 햇살 아래 풀밭에서 함께 모여 있는 모습이 매우 평화롭습니다.\n\n## 🐰 토끼의 주요 특징\n\n### 외형적 특징\n- **긴 귀**: 체온 조절과 주변 소리 감지에 사용\n- **강한 뒷다리**: 도약 능력 (최대 3m 이상 점프 가능)\n- **부드러운 털**: 다양한 색상 (흰색, 갈색, 회색 등)\n- **큰 눈**: 시야가 넓고 약 360도 시야 가능\n- **코끝이 항상 움직임**: 뛰어난 후각\n\n### 생태적 특징\n- **초식동물**: 풀, 잎, 채소 등을 주식으로 함\n- **야행성**: 주로 새벽과 저녁에 활동\n- **번식력**: 한 번에 4~12마리의 새끼를 낳음\n- **수명**: 야생에서는 약 1~2년, 가축으로는 8~12년\n\n### 행동적 특징\n- 사회성이 강해 무리 생활을 선호\n- 위험을 감지하면 뒷다리로 땅을 쳐서 경고\n- 청소를 직접 하는 습성이 있음\n- 연속적으로 도약하여 이동 (로핑 행동)\n\n## 🌸 이미지의 토끼\n- 큰 토끼 한 마리와 작은 새끼 토끼 여러 마리가 함께 있는 모습\n- **모계(어미-새끼) 가족 단위**로 행동하는 토끼의 특성을 잘 보여줌\n- 따뜻한 갈색과 연한 색상의 털을 가진 품종으로 보임\n\n토끼는 전 세계적으로 애완동물로도 인기가 많으며, 중국 등 동양 문화에서는 **달의 동물**로 여겨져 풍요와 번영의 상징으로 인식됩니다. 🐇✨'

#### 3. OCR

In [20]:
# 모든 이미지를 jpeg 저장 / 인코딩

def pil_to_base64(img:Image,format="JPEG")->str:
    buffer = BytesIO()
    img.save(buffer, format=format)
    return base64.b64encode(buffer.getvalue()).decode('utf-8')

In [24]:
img = Image.open("./image/system.png")

# resize
img_resized = img.resize((800,600))

# Gray
img_gray = img.convert('L')
img_b64 = pil_to_base64(img_gray)

message = HumanMessage(
    content= [
        {'type':"text", "text":" 이 문서의 텍스트를 추출해주세요."},
        {"type":"image_url","image_url":{'url':f'data:image/jpeg;base64,{img_b64}'}},
    ]
)

result=vision_llm.invoke([message]).content
print(result)

# 추출된 텍스트

## 미래로봇추진단(서울 근무)
## S/W개발 - 시스템 소프트웨어

---

### 포지션 소개 (Job Overview)
휴머노이드 로봇의 실시간 제어 시스템 및 소프트웨어 플랫폼을 개발하는 직무입니다. 운영체제 환경 구성, 디바이스 드라이버, 실시간 제어 프레임워크 등 로봇 동작의 핵심 기반이 되는 소프트웨어를 설계 및 개방하며, 하드웨어와 설계 조직 및 AI 연구 조직과 긴밀히 협업합니다.

---

### 수행업무 (Job Details)
- 휴머노이드 로봇의 실시간 제어 프레임워크를 설계하고 개발합니다.
- 모터, 센서 등 로봇 하드웨어 제어를 위한 디바이스 드라이버 및 하드웨어 추상화 계층을 개발합니다.
- 로봇 제어를 운영체제(Linux 기반 실시간 OS 등) 환경을 구성하고 시스템 성능을 최적화합니다.
- 유관 부서와 협업하여 로봇 조작 등 AI 기능과 제어 시스템 간 연동 미들웨어를 개발합니다.

---

### 자격요건 (Requirements)
- 컴퓨터, 전기 전자, 기계, 로봇공학 등 관련 전공을 하신 분
- 운영체제 기반 개방에 대한 이해도를 보유하신 분
- 요구사항을 분석하여 소프트웨어를 구조적으로 설계 및 구현하는 역량을 보유하신 분
- 다양한 분야의 엔지니어와 적극적으로 소통하며 협업할 수 있는 역량을 보유하신 분

---

### 우대사항 (Preferences)
- C/C++ 기반 시스템 프로그래밍 역량을 보유하신 분
- Linux 기반 임베디드 시스템 개발 경력을 보유하신 분 (Kernel, Device Driver, BSP 등)
- CAN, EtherCAT, UDP 등 하드웨어 통신 프로토콜 활용 경험을 보유하신 분
- ROS2 기반 로봇버스 소프트웨어 수행 경력을 보유하신 분
- Git 기반 협업 및 CI/CD 환경에서의 개발 경험을 보유하신 분

---

### 커리어 비전 (Career Vision)
휴머노이드 로봇의 초기 개발 단계부터 참여하여 핵심 개발자로 성장할 수 있습니다. 실시간 제어,

In [25]:
def compare_images(image_paths,question):
    content =[]

    for i, path in enumerate(image_paths,1):
        img_b64 = encode_image(path)
        content.append({"type":"image_url","image_url":{'url':f'data:image/jpeg;base64,{img_b64}'}})
        content.append({'type':"text", "text":f"[이미지 {i}]"})

    # 질문
    content.append({'type':"text", "text":question})

    message = HumanMessage(content= content)

    response = vision_llm.invoke([message])
    
    return response.content


In [27]:
result = compare_images(['./image/fridge.jpg', './image/table.jpg'],'두 제품의 차이점을 비교 분석해줘')
print(result)

# 두 제품 비교 분석

두 이미지는 **완전히 다른 종류의 가전/가구 제품**이므로, 각 제품의 특징을 먼저 살펴본 후 비교하겠습니다.

---

## 📌 제품 1: 4도어 냉장고

| 항목 | 내용 |
|------|------|
| **종류** | 양문형 4도어 냉장고 |
| **색상** | 화이트/크림 |
| **구조** | 상단 2도어 + 중간 손잡이 바 + 하단 2도어 |
| **디자인** | 모던하고 깔끔한 미니멀 스타일 |
| **소재** | 금속 외함 (스테인리스 코팅 추정) |
| **용도** | 식품 보관·냉동·냉장 |
| **배치** | 부엌 싱크대/벽면 일체형 배치 |

---

## 📌 제품 2: 식탁 (다이닝 테이블)

| 항목 | 내용 |
|------|------|
| **종류** | 직사각형 다이닝 테이블 |
| **상판** | 화이트 대리석 패턴 (그레이 베인) |
| **프레임** | 블랙 금속 다리 |
| **다리 팁** | 골드/브라스 컬러 장식 |
| **디자인** | 모던 럭셔리 + 북유럽 미니멀 |
| **소재** | 엔지니어드 스톤/MDF + 강철 |
| **용도** | 식사·작업·소품 연출 |
| **배치** | 거실/다이닝 공간 중심 |

---

## 🔍 핵심 차이점

### 1️⃣ **용도(Category)**
- 냉장고 → **가전제품(대형)** / 보존·저장 기능
- 식탁 → **가구(중형)** / 사용·활동 기능

### 2️⃣ **공통 디자인 언어**
- 두 제품 모두 **모던 미니멀 + 화이트톤 + 블랙 액센트**라는 동일한 인테리어 트렌드를 공유
- 서로 다른 공간(부엌 vs 다이닝)에서도 **스타일 통일성**이 있어 같은 집에 배치 시 조화 가능

### 3️⃣ **컬러 팔레트**
- **냉장고**: 무채색 화이트 + 블랙 라인
- **식탁**: 화이트 + 그레이 + 블랙 + 골드
- 식탁이 더 **다양한 컬러 포인트**(골드)를 가짐 → 시각적 강조 효과

### 4️⃣ **소재 대비**
- 

In [29]:
result = compare_images(['./image/chart1.png', './image/chart2.png'],'두 차트를 비교라여 주요 변화 추치를 분석해줘')
print(result)

# 두 차트 비교 분석: 화장품 수출 추이

## 📊 차트 1: 최근 6개월 전체 수출 추이 (25년 12월 ~ 26년 5월)

| 기간 | 수출액(백만$) | YoY |
|------|---------------|------|
| 25년 12월 | 883.7 | +10.2% |
| 26년 1월 | 841.6 | **+33.7%** |
| 26년 2월 | 752.1 | +1.7% |
| 26년 3월 | 960.4 | +23.0% |
| 26년 4월 | **1,096.3** | +21.7% |
| 26년 5월(1-20일) | 671.1 | **-16.0%** |

## 📊 차트 2: 5월 1~20일 주요 국가별 일평균 수출액

| 국가 | 일평균 수출액(백만$) | YoY |
|------|----------------------|------|
| 중화권(중국) | **13.13** | **+76.4%** |
| 미국 | 11.66 | +40.3% |
| 유럽 | 11.14 | +61.3% |
| 동남아 | 4.91 | +22.0% |
| 일본 | 약 3.2 | **-14.1%** |

---

## 🔍 주요 변화 추이 분석

### 1️⃣ 전체 흐름: 고성장 → 급격한 둔화
- 26년 1월(+33.7%)부터 4월(+21.7%)까지 **꾸준한 두 자릿수 성장세**를 유지
- 4월에 **역대 최고 수출액 1,096.3백만$** 기록
- 그러나 **5월 들어 YoY -16%** 로 급반전, 6개월 만에 첫 마이너스 성장

### 2️⃣ 5월 마이너스 성장의 원인: 일본시장의 급감
- 5대 주요국 중 **일본만 유일한 역성장(-14.1%)**
- 일평균 수출액도 5대국 중 **최하위 수준**으로 추락
- 그 외 4개국(중국·미국·유럽·동남아)은 모두 **+22% 이상 고성장**

### 3️⃣ 신규 성장동력: 중화권(중국)의 약진
- **일평균 수출액 1위(13.13백만$) + 최대 성장률(+76.4%)**
- 미국·유럽도 40~60%대 고성장 지속
- → 대중국 의

### 멀티 모달
- 이미지 기반 문서 분석 시스템

In [ ]:
# 이미지 => 설명문

def image_to_caption(image_path):
    img_b64 = encode_image(image_path)
    message = HumanMessage(
        content= [ 
            {"type":"image_url","image_url":{'url':f'data:image/jpeg;base64,{img_b64}'}},
            {'type':"text", "text":"""
            이 이미지를 검색용 설명문으로 요약하세요.
             200자 이낼 작성하세요.
             핵심 객체와 텍스트만 포함하세요.
            """
            }
        ]
    )
    return vision_llm.invoke([message]).content

In [7]:
caption = image_to_caption("./image/chart1.png")

caption

'최근 6개월 전체 화장품 수출액 및 YoY 추이 차트. 25년 12월 883.7백만달러(10.2%), 26년 1월 841.6(33.7%), 2월 752.1(1.7%), 3월 960.4(23%), 4월 1096.3(21.7%), 5월(1-20일) 671.1(-16%)로 4월 정점 후 5월 급감. 파란 막대 수출액, 빨간 선 전년동기대비 증가율.'

In [29]:
# Document

def build_multimodal_index(image_dir:str, text_docs:list[Document]):
    all_docs = list(text_docs)

    # image_dir 안 파일 가져오기
    image_file = list(Path(image_dir).glob("*.{jpg,jpeg,png}"))

    # caption 생성 => Document => 임베딩
    for img_path in image_file:
        caption = image_to_caption(image_path=img_path)
        
        doc = Document(page_content=caption,metadata={
            "source":str(img_path),
            "type": "image",
            "image_path":str(img_path)
        })
        all_docs.append(doc)

    return Chroma.from_documents(all_docs,watsonx_embedding,persist_directory="./db/multimodal_db")


In [21]:
# 검색 결과 이미지 포함 여부 확인
def search_with_images(vectorstore, query):
    results = vectorstore.similarity_search(query, k= 5)
    text_results = [r for r in results if r.metadata.get("type") != 'image']
    image_results = [r for r in results if r.metadata.get("type") == 'image']
    print(f"텍스트 결과: {len(text_results)}개, 이미지 결과: {len(image_results)}개")

    return text_results, image_results

In [23]:
image_doc = Document(
    page_content=caption,
    metadata={"type":"image","image_path":"./image/chart1.png"}
)
docs = [
    Document(
        page_content="2025년 매출은 증가했다."
    ),
    image_doc
]

build_multimodal_index("./image",docs)

In [30]:
vectorstore = Chroma(embedding_function=watsonx_embedding, persist_directory="./db/multimodal_db")

results = vectorstore.similarity_search("매출 추세", k=3)

for r in results:
    print(r.page_content)

2025년 매출은 증가했다.
2025년 매출은 증가했다.
최근 6개월 전체 화장품 수출액 및 YoY 추이 차트. 25년 12월 883.7백만달러(10.2%), 26년 1월 841.6(33.7%), 2월 752.1(1.7%), 3월 960.4(23%), 4월 1096.3(21.7%), 5월(1-20일) 671.1(-16%)로 4월 정점 후 5월 급감. 파란 막대 수출액, 빨간 선 전년동기대비 증가율.


In [24]:
def multimodal_answer(vectorstore,question):
    text_results, image_results = search_with_images(vectorstore=vectorstore,query=question)

    # 텍스트 결과 하나의 컨텍스트로 생성
    text_context = "\n\n".join(r.page_content for r in text_results)
    
    # 이미지로 검색된 경우
    image_context = ""
    referenced_images=[]
    for img_doc in image_results[:3]:
        img_path = img_doc.metadata.get("image_path")

        img_b64 = encode_image(img_path)
        analysis=vision_llm.invoke([
            HumanMessage(
                content= [ 
                    {"type":"image_url","image_url":{'url':f'data:image/jpeg;base64,{img_b64}'}},
                    {'type':"text", "text":"이 이미지에서 다음 질문과 관련된 내용을 설명하세요: {question}."}
                ]
            )
        ]).content

        image_context += f"[이미지 분석: {img_path}]\n{analysis}\n\n"
        referenced_images.append(img_path)

    # 최종 답변
    combined_context = text_context + "\n\n" + image_context

    final_prompt = ChatPromptTemplate.from_messages([
        ("system", "다음 텍스트와 이미지 분석 결과를 참고하여 질문에 답하세요\n\n{context}"),  
        ("human", "{question}"),
    ])

    parser = StrOutputParser()

    chain = final_prompt | watson_llm | parser

    answer = chain.invoke({
        "context":combined_context,
        "question":question
    })

    return {"answer":answer, "images": referenced_images}

In [31]:
multimodal_answer(vectorstore, "최근 6개월 전체 화장품 수출액은?")

텍스트 결과: 3개, 이미지 결과: 1개


{'answer': '최근 6개월 전체 화장품 수출액은 다음과 같습니다:\n\n- 2025년 12월: 883.7백만달러\n- 2026년 1월: 841.6백만달러\n- 2026년 2월: 752.1백만달러\n- 2026년 3월: 960.4백만달러\n- 2026년 4월: 1,096.3백만달러\n- 2026년 5월(1-20일): 671.1백만달러\n\n이 데이터는 최근 6개월 동안의 화장품 수출액을 나타냅니다.',
 'images': ['./image/chart1.png']}